# Ungraded Lab: GradCAM (PyTorch)

This lab will walk you through generating gradient-weighted class activation maps (GradCAMs) for model predictions.
- This is similar to the CAMs you generated before except:
  - GradCAMs uses gradients instead of the global average pooling weights to weight the activations.

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. The intermediate-layer "visualization model" and the gradient tape are replaced by PyTorch forward hooks and `torch.autograd.grad`.

## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import cv2

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Download and Prepare the Dataset

You will use the Cats vs Dogs dataset again for this exercise. The following will prepare the train, test, and eval sets.

In [ ]:
import os
import urllib.request
import zipfile

data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
data_file_name = "data/kagglecatsanddogs_5340.zip"
os.makedirs("data", exist_ok=True)

if not os.path.exists(data_file_name):
    print("downloading the dataset (about 800 MB)...")
    urllib.request.urlretrieve(data_url, data_file_name)
if not os.path.exists("data/catsdogs/PetImages"):
    with zipfile.ZipFile(data_file_name, 'r') as zip_ref:
        zip_ref.extractall("data/catsdogs/")
print("dataset ready")

In [ ]:
import platform
from PIL import Image, ImageFile

# DataLoader worker processes on macOS are started with "spawn", which cannot see classes defined
# inside a notebook (such as the Dataset below). "fork" works fine for the image decoding the workers do.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None

# a few images in this dataset are truncated; let PIL load what it can instead of raising
ImageFile.LOAD_TRUNCATED_IMAGES = True


def list_cats_vs_dogs_files(root="data/catsdogs/PetImages", cache="data/catsdogs/file_list.txt"):
    '''
    Returns a sorted list of (path, label) pairs, label 0 for cats and 1 for dogs.

    Files that cannot be decoded as images are skipped (like tfds does when it prepares the
    dataset). Checking all 25,000 files takes a minute, so the result is cached on disk.

    Args:
      root (string) -- folder holding the Cat and Dog subfolders
      cache (string) -- file the resulting list is cached in

    Returns:
      list -- (path, label) pairs, sorted and stable across runs
    '''
    if os.path.exists(cache):
        with open(cache) as f:
            return [(line.split("\t")[0], int(line.split("\t")[1])) for line in f.read().splitlines()]

    files = []
    for label, folder in enumerate(["Cat", "Dog"]):
        for name in sorted(os.listdir(os.path.join(root, folder))):
            path = os.path.join(root, folder, name)
            try:
                with Image.open(path) as img:
                    img.verify()
                files.append((path, label))
            except Exception:
                pass   # not a valid image
    with open(cache, "w") as f:
        f.write("\n".join(f"{p}\t{l}" for p, l in files))
    return files


def take_split(files, start, end):
    '''
    The equivalent of the tfds split syntax train[start:end], with the bounds as fractions.

    Args:
      files (list) -- the full list of (path, label) pairs
      start (float) -- fraction of the list to start at, 0.0 is the beginning
      end (float) -- fraction of the list to stop at, 1.0 is the end

    Returns:
      list -- the selected slice of (path, label) pairs
    '''
    n = len(files)
    return files[int(start * n):int(end * n)]


class CatsVsDogs(Dataset):
    '''yields (image, label) pairs; `transform` turns the PIL image into a tensor'''

    def __init__(self, files, transform):
        '''
        Stores the (path, label) pairs and the transform applied to each image.

        Args:
          files (list) -- (path, label) pairs, label 0 for cat and 1 for dog
          transform (callable) -- turns a PIL image into a tensor
        '''
        self.files = files
        self.transform = transform

    def __len__(self):
        '''
        Reports how many items this split holds.

        Returns:
          int -- number of images in this split
        '''
        return len(self.files)

    def __getitem__(self, idx):
        '''
        Loads image `idx`, forces it to RGB, and applies the transform.

        Args:
          idx (int) -- index of the image to fetch

        Returns:
          (tensor, int) -- the transformed image and its label
        '''
        path, label = self.files[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), label

In [ ]:
files = list_cats_vs_dogs_files()

# load the dataset given the splits: train[:80%], train[80%:90%], train[90%:]
train_examples = take_split(files, 0.0, 0.8)
validation_examples = take_split(files, 0.8, 0.9)
test_examples = take_split(files, 0.9, 1.0)

num_examples = len(files)
num_classes = 2

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# resizes the image and normalizes the pixel values to [0, 1]
format_image = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])

# prepare batches
train_batches = DataLoader(CatsVsDogs(train_examples, format_image), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)
validation_batches = DataLoader(CatsVsDogs(validation_examples, format_image), batch_size=BATCH_SIZE, num_workers=4, multiprocessing_context=MP_CONTEXT, persistent_workers=True)
test_batches = DataLoader(CatsVsDogs(test_examples, format_image), batch_size=1)

## Modelling

You will use a pre-trained VGG16 network as your base model for the classifier. This will be followed by a global average pooling (GAP) and a 2-neuron linear layer for the output (the softmax is applied by the loss). The earlier VGG blocks will be frozen and we will just fine-tune the final layers during training. These steps are shown in the utility function below.

The torchvision VGG16 weights expect inputs normalized with the ImageNet mean and standard deviation, so the model starts with a `Normalize` step. Keeping the normalization *inside* the model means the images in the batches stay in the `[0, 1]` range, which is convenient for plotting them later.

In [ ]:
class Normalize(nn.Module):
    '''ImageNet mean/std normalization as a layer'''
    def __init__(self):
        '''
        Registers the ImageNet channel mean and standard deviation as buffers.
        '''
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        '''
        Applies the ImageNet normalization to a batch of images.

        Args:
          x (tensor) -- batch of images in [0, 1], shape (N, 3, H, W)

        Returns:
          tensor -- the batch normalized with the ImageNet statistics
        '''
        return (x - self.mean) / self.std


def build_model(device):
  '''
  Builds the VGG16 classifier, freezing every block except block 5.

  Args:
    device (torch.device) -- device the model is moved to

  Returns:
    (nn.Module, Optimizer, callable) -- the model, its RMSprop optimizer and the loss function
  '''
  # load the base VGG16 model (the convolutional part only, i.e. include_top=False)
  base_model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features

  model = nn.Sequential(
      Normalize(),
      base_model,
      # add a GAP layer
      nn.AdaptiveAvgPool2d(1),
      nn.Flatten(),
      # output has two neurons for the 2 classes (cats and dogs)
      nn.Linear(512, 2),
  ).to(device)

  # freeze the earlier layers: everything except the three convolutions of block 5
  # (the equivalent of freezing base_model.layers[:-4] in Keras)
  for param in base_model.parameters():
      param.requires_grad = False
  for idx in [24, 26, 28]:   # block5_conv1, block5_conv2, block5_conv3
      for param in base_model[idx].parameters():
          param.requires_grad = True

  # choose the optimizer (only the trainable parameters are passed to it)
  # alpha/eps are set to the Keras RMSprop defaults (rho=0.9, epsilon=1e-7); PyTorch's defaults (0.99, 1e-8) take much larger first steps
  optimizer = torch.optim.RMSprop(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001, alpha=0.9, eps=1e-7)

  # configure the model for training
  loss_fn = nn.CrossEntropyLoss()   # sparse_categorical_crossentropy

  # display the summary
  summary(model, input_size=(1, 3) + IMAGE_SIZE, device=device)

  return model, optimizer, loss_fn

In [ ]:
model, optimizer, loss_fn = build_model(device)

You can now train the model. This will take a while to run.

In [ ]:
EPOCHS = 3


def run_epoch(loader, model, loss_fn, optimizer, device, train):
    '''
    Runs one pass over `loader`, training or evaluating.

    Args:
      loader (DataLoader) -- yields (images, labels) batches
      model (nn.Module) -- classifier being trained or evaluated
      loss_fn (callable) -- loss applied to (logits, labels)
      optimizer (Optimizer) -- updates weights; only used when train is True
      device (torch.device) -- device the batches are moved to
      train (bool) -- True updates the weights, False only measures

    Returns:
      (float, float) -- mean loss and accuracy over the epoch
    '''
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = loss_fn(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(images)
    return total_loss / count, correct / count


for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_batches, model, loss_fn, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(validation_batches, model, loss_fn, optimizer, device, train=False)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

## Model Interpretability

Let's now go through the steps to generate the class activation maps. You will start by specifying the layers you want to visualize.

In Keras you would build a second `Model` that returns the outputs of several layers. In PyTorch you attach *forward hooks* to the layers instead: each hook stores the layer's output when the model runs. The Keras `Conv2D` layers include their ReLU activation, so the hooks are placed on the `ReLU` module that follows each convolution.

In [ ]:
vgg_features = model[1]

# (index in vgg_features, Keras-style name) for every conv (after its ReLU) and pooling layer,
# except the very last pooling layer (Keras: model.layers[1:18])
layer_specs = [(1, 'block1_conv1'), (3, 'block1_conv2'), (4, 'block1_pool'),
               (6, 'block2_conv1'), (8, 'block2_conv2'), (9, 'block2_pool'),
               (11, 'block3_conv1'), (13, 'block3_conv2'), (15, 'block3_conv3'), (16, 'block3_pool'),
               (18, 'block4_conv1'), (20, 'block4_conv2'), (22, 'block4_conv3'), (23, 'block4_pool'),
               (25, 'block5_conv1'), (27, 'block5_conv2'), (29, 'block5_conv3')]

# store the layer names we are interested in
layer_names = [name for _, name in layer_specs]

# forward hooks that record each layer's output (moved to channels-last numpy arrays for plotting)
_activations = {}
def make_hook(name, store):
    '''
    Builds a forward hook that records one layer's output into `store`.

    Args:
      name (string) -- key the activation is stored under
      store (dict) -- dictionary the hook writes into

    Returns:
      callable -- hook suitable for register_forward_hook
    '''
    def hook(module, inputs, output):
        '''
        Records the layer output channels-last, detached from the graph.

        Args:
          module (nn.Module) -- the layer the hook is attached to
          inputs (tuple) -- the layer's inputs, unused here
          output (tensor) -- the layer's output, which is what gets stored
        '''
        store[name] = output.detach().permute(0, 2, 3, 1).cpu().numpy()
    return hook

for idx, name in layer_specs:
    vgg_features[idx].register_forward_hook(make_hook(name, _activations))


def vis_model_predict(image_batch, model, device, layer_names, store):
    '''
    Runs the model so the hooks fire, then returns the recorded activations.

    Args:
      image_batch (tensor) -- batch of images, shape (N, 3, 224, 224)
      model (nn.Module) -- the model the hooks are attached to
      device (torch.device) -- device to run on
      layer_names (list of str) -- names to return, in layer order
      store (dict) -- dictionary the hooks write into

    Returns:
      list of arrays -- one activation array per layer, channels-last
    '''
    model.eval()
    with torch.no_grad():
        model(image_batch.to(device))
    return [store[name] for name in layer_names]


print("Layers that will be used for visualization: ")
print(layer_names)

### Class activation maps (GradCAM)

We'll define a few more functions to output the maps. `get_CAM()` is the function highlighted in the lectures and takes care of generating the heatmap of gradient weighted features. `show_random_sample()` takes care of plotting the results.

Instead of a `GradientTape`, the conv layer output is captured with a hook and `torch.autograd.grad` computes the gradient of the loss with respect to it.

In [ ]:
def get_CAM(processed_image, actual_label, model, device, layer_specs, layer_name='block5_conv3'):
    '''
    Computes the gradient-weighted class activation map for a single image.

    Args:
      processed_image (tensor) -- one preprocessed image, shape (1, 3, 224, 224)
      actual_label (int) -- ground truth label, 0 for cat and 1 for dog
      model (nn.Module) -- the trained classifier
      device (torch.device) -- device the model runs on
      layer_specs (list) -- (index, name) pairs identifying the conv layers
      layer_name (string) -- which conv layer the map is built from

    Returns:
      array -- heatmap of shape (14, 14) scaled to [0, 1]
    '''
    layer_idx = dict((name, idx) for idx, name in layer_specs)[layer_name]

    # capture the conv layer output *with* its computation graph so we can differentiate through it
    captured = {}
    def hook(module, inputs, output):
        '''
        Captures the conv layer output, keeping it attached to the autograd graph.

        Args:
          module (nn.Module) -- the layer the hook is attached to
          inputs (tuple) -- the layer's inputs, unused here
          output (tensor) -- the layer's output, which is what gets captured
        '''
        captured['conv_output_values'] = output
    handle = model[1][layer_idx].register_forward_hook(hook)

    model.eval()
    predictions = torch.softmax(model(processed_image.to(device)), dim=1)
    handle.remove()
    conv_output_values = captured['conv_output_values']

    ## Use binary cross entropy loss
    ## actual_label is 0 if cat, 1 if dog
    # get prediction probability of dog
    # If model does well,
    # pred_prob should be close to 0 if cat, close to 1 if dog
    pred_prob = predictions[:,1]

    # make sure actual_label is a float, like the rest of the loss calculation
    actual_label = torch.as_tensor(actual_label, dtype=torch.float32, device=device)

    # add a tiny value to avoid log of 0
    smoothing = 0.00001

    # Calculate loss as binary cross entropy
    loss = -1 * (actual_label * torch.log(pred_prob + smoothing) + (1 - actual_label) * torch.log(1 - pred_prob + smoothing))
    print(f"binary loss: {loss.item()}")

    # get the gradient of the loss with respect to the outputs of the last conv layer
    grads_values = torch.autograd.grad(loss.sum(), conv_output_values)[0]
    # average the gradients over the batch, height and width -> one weight per channel
    grads_values = grads_values.mean(dim=(0, 2, 3))

    # to channels-last numpy: (512, 14, 14) -> (14, 14, 512)
    conv_output_values = np.squeeze(conv_output_values.detach().cpu().numpy()).transpose(1, 2, 0).copy()
    grads_values = grads_values.cpu().numpy()

    # weight the convolution outputs with the computed gradients
    for i in range(512):
        conv_output_values[:,:,i] *= grads_values[i]
    heatmap = np.mean(conv_output_values, axis=-1)

    heatmap = np.maximum(heatmap, 0)
    heatmap /= heatmap.max()

    del conv_output_values, grads_values, loss

    return heatmap

In [ ]:
def show_sample(model, device, test_batches, layer_specs, layer_names, store, idx=None):
    '''
    Plots one test image alongside a feature map, its CAM, and the two overlaid.

    Args:
      model (nn.Module) -- the trained classifier
      device (torch.device) -- device the model runs on
      test_batches (DataLoader) -- test set the sample is drawn from
      layer_specs (list) -- (index, name) pairs identifying the conv layers
      layer_names (list of str) -- layer names in order
      store (dict) -- dictionary the forward hooks write into
      idx (int) -- index of the image to show, or None for a random one

    Returns:
      list of arrays -- intermediate activations of the chosen image
    '''
    # if image index is specified, get that image
    if idx:
        sample_image, sample_label = test_batches.dataset[idx]
    # otherwise if idx is not specified, get a random image
    else:
        sample_image, sample_label = test_batches.dataset[np.random.randint(len(test_batches.dataset))]

    sample_image_processed = sample_image.unsqueeze(0)      # add the batch dimension: (1, 3, 224, 224)

    activations = vis_model_predict(sample_image_processed, model, device, layer_names, store)

    model.eval()
    with torch.no_grad():
        pred_label = model(sample_image_processed.to(device)).argmax(dim=-1).item()

    sample_activation = activations[0][0,:,:,16]

    sample_activation-=sample_activation.mean()
    sample_activation/=sample_activation.std()

    sample_activation *=255
    sample_activation = np.clip(sample_activation, 0, 255).astype(np.uint8)

    heatmap = get_CAM(sample_image_processed, sample_label, model, device, layer_specs)
    heatmap = cv2.resize(heatmap, (sample_image.shape[2], sample_image.shape[1]))
    heatmap = heatmap *255
    heatmap = np.clip(heatmap, 0, 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_HOT)
    converted_img = sample_image.permute(1, 2, 0).numpy()    # (224, 224, 3) in [0, 1]
    super_imposed_image = np.clip(cv2.addWeighted(converted_img, 0.8, heatmap.astype('float32'), 2e-3, 0.0), 0, 1)

    f,ax = plt.subplots(2,2, figsize=(15,8))

    ax[0,0].imshow(converted_img)
    ax[0,0].set_title(f"True label: {sample_label} \n Predicted label: {pred_label}")
    ax[0,0].axis('off')

    ax[0,1].imshow(sample_activation)
    ax[0,1].set_title("Random feature map")
    ax[0,1].axis('off')

    ax[1,0].imshow(heatmap)
    ax[1,0].set_title("Class Activation Map")
    ax[1,0].axis('off')

    ax[1,1].imshow(super_imposed_image)
    ax[1,1].set_title("Activation map superimposed")
    ax[1,1].axis('off')
    plt.tight_layout()
    plt.show()

    return activations

### Time to visualize the results

In [ ]:
# Choose an image index to show, or leave it as None to get a random image
activations = show_sample(model, device, test_batches, layer_specs, layer_names, _activations, idx=None)

### Intermediate activations of layers

You can use the utility function below to visualize the activations in the intermediate layers you defined earlier. This plots the feature side by side for each convolution layer starting from the earliest layer all the way to the final convolution layer.

In [ ]:
def visualize_intermediate_activations(layer_names, activations):
    '''
    Plots every feature map of each recorded layer as a grid.

    Args:
      layer_names (list of str) -- names of the layers, in order
      activations (list of arrays) -- matching activation arrays, channels-last
    '''
    assert len(layer_names)==len(activations), "Make sure layers and activation values match"
    images_per_row=16

    for layer_name, layer_activation in zip(layer_names, activations):
        nb_features = layer_activation.shape[-1]
        size= layer_activation.shape[1]

        nb_cols = nb_features // images_per_row
        grid = np.zeros((size*nb_cols, size*images_per_row))

        for col in range(nb_cols):
            for row in range(images_per_row):
                feature_map = layer_activation[0,:,:,col*images_per_row + row].copy()
                feature_map -= feature_map.mean()
                feature_map /= feature_map.std() + 1e-8
                feature_map *=255
                feature_map = np.clip(feature_map, 0, 255).astype(np.uint8)

                grid[col*size:(col+1)*size, row*size:(row+1)*size] = feature_map

        scale = 1./size
        plt.figure(figsize=(scale*grid.shape[1], scale*grid.shape[0]))
        plt.title(layer_name)
        plt.grid(False)
        plt.axis('off')
        plt.imshow(grid, aspect='auto', cmap='viridis')
    plt.show()

In [ ]:
visualize_intermediate_activations(activations=activations,
                                   layer_names=layer_names)

If you scroll all the way down to see the outputs of the final conv layer, you'll see that there are very few active features and these are mostly located in the face of the cat. This is the region of the image that your model looks at when determining the class.